In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# React to a store event and publish a recommendation

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

Not every agent is a chat. An ambient agent runs when something happens: a shelf scan finds no stock, a pickup backlog passes a threshold, a loss pattern appears. It reads the facts, decides what to recommend, and sends the recommendation somewhere a person will see it.

### Event in, recommendation out

The input to this agent is a JSON event, here an on-shelf availability exception: a scan found none of a product on the shelf. The agent reads the stock position at that store with `check_store_stock`, drafts one recommendation with the counts as its reason, and publishes it. It never creates a task; the store manager decides.

### Publishing with the Pub/Sub toolset

ADK includes a [Pub/Sub toolset](https://adk.dev/integrations/pubsub/). This agent is given only its `publish_message` tool, so publishing is the one thing it can do outside the store data. Anything subscribed to the topic receives the recommendation: a task queue, a notification service, or the pull subscription you read in this notebook.

<img width="60%" src="../../docs/diagrams/q11.png" alt="An exception event goes to the agent, which reads stock and publishes a recommendation to Pub/Sub" />

### Objectives

In this tutorial, you will learn how to build an ADK agent that is triggered by an event rather than a person.

You will complete the following tasks:

- Create a Pub/Sub topic and subscription for your recommendations
- Send an on-shelf availability event to the agent
- Read the recommendation it published
- Send an event type the agent does not handle

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery
- Pub/Sub

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing) and [Pub/Sub pricing](https://cloud.google.com/pubsub/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded in the workshop notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The quickstart imports shared store code from the repository root, two folders up
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agents" / "cymbal_store_ops").is_dir())
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the output to the agent's own events: ADK marks experimental and deprecated features
# with warnings and logs configuration hints, and the Gen AI SDK logs a note whenever a
# response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
import json

from google.adk.runners import InMemoryRunner
from google.api_core.exceptions import AlreadyExists, NotFound
from google.cloud import pubsub_v1
from google.genai import types

## Create the recommendations topic

The topic and subscription names include your namespace, so two people in the same project never read each other's recommendations. Running the cell again is safe: an existing topic or subscription is kept.

In [5]:
TOPIC_NAME = f"cymbal-store-ops-recommendations-{WORKSHOP_NAMESPACE}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_NAME)
subscription_path = subscriber.subscription_path(PROJECT_ID, f"{TOPIC_NAME}-pull")

try:
    publisher.create_topic(name=topic_path)
except AlreadyExists:
    pass
try:
    subscriber.create_subscription(name=subscription_path, topic=topic_path)
except AlreadyExists:
    pass

print(topic_path)
print(subscription_path)

projects/mattrobn-sandbox/topics/cymbal-store-ops-recommendations-opsreview
projects/mattrobn-sandbox/subscriptions/cymbal-store-ops-recommendations-opsreview-pull


## Load the agent

The agent reads the topic from the `RECOMMENDATIONS_TOPIC` environment variable when it is built, so set it before importing `agent.py`.

In [6]:
os.environ["RECOMMENDATIONS_TOPIC"] = topic_path

from agent import app

print(app.root_agent.instruction)

You receive a store exception event as JSON with event_type, store_id, product_name and detected_at.
This agent handles event_type "osa_exception" (nothing on the shelf for a product). For any other event type, publish
nothing and reply that the event type is not handled here.
1. Call check_store_stock with the event's product_name and store_id.
2. Draft ONE recommendation from the tool result: the action is the tool's `recommendation` (backroom_check |
   replenish | cycle_count | escalate), and the rationale is one line with the on-shelf, backroom and on-hand counts.
3. Publish it with publish_message to the topic projects/mattrobn-sandbox/topics/cymbal-store-ops-recommendations-opsreview as JSON with the keys store_id, product_id, action, rationale
   and detected_at.
4. Reply in one sentence: the action, the store and the product, and that it was published for the manager to approve.
You recommend; the store manager decides. Never claim a task was created or stock was moved.


## Send an event

There is no signed-in person: the event carries its own `store_id`. Send the event as the message text and print what the agent does.

In [7]:
runner = InMemoryRunner(app=app)


async def send_event(event: dict) -> None:
    """Send one event to the agent in a new session and print its tool calls and reply."""
    session = await runner.session_service.create_session(app_name=app.name, user_id="shelf-scanner")
    message = types.Content(role="user", parts=[types.Part(text=json.dumps(event))])
    async for agent_event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in agent_event.get_function_calls():
            print(f"calls {call.name}")
        if agent_event.is_final_response() and agent_event.content and agent_event.content.parts:
            print("\n" + "".join(part.text or "" for part in agent_event.content.parts if not part.thought))

In [8]:
await send_event(
    {
        "event_type": "osa_exception",
        "store_id": "S-014",
        "product_name": "Lumière Hydra Cream",
        "detected_at": "2026-10-03T08:55:00-05:00",
    }
)

calls check_store_stock


calls publish_message



A backroom check recommendation for Lumière Hydra Cream at store S-014 has been published for the store manager to approve.


## Read the recommendation

Pull from the subscription to see what the agent published. Acknowledging the messages removes them from the subscription.

In [9]:
response = subscriber.pull(subscription=subscription_path, max_messages=10, timeout=30)

for received in response.received_messages:
    print(json.dumps(json.loads(received.message.data), indent=2))

if response.received_messages:
    subscriber.acknowledge(
        subscription=subscription_path,
        ack_ids=[received.ack_id for received in response.received_messages],
    )

{
  "store_id": "S-014",
  "product_id": "P-0101",
  "action": "backroom_check",
  "rationale": "On-shelf count is 0, backroom count is 7, and on-hand count is 7.",
  "detected_at": "2026-10-03T08:55:00-05:00"
}


The message has the five keys from the instruction. The action, `backroom_check`, comes from the stock tool's recommendation, and the rationale carries the counts: 0 on the shelf, 7 in the backroom, 7 on hand. `detected_at` is copied from the event.

## Send an event the agent does not handle

The instruction limits the agent to `osa_exception` events. For anything else it publishes nothing and says so:

In [10]:
await send_event(
    {
        "event_type": "price_change",
        "store_id": "S-014",
        "product_name": "Lumière Hydra Cream",
        "detected_at": "2026-10-03T08:55:00-05:00",
    }
)


This event type ("price_change") is not handled here. This agent only processes "osa_exception" events.


## Cleaning up

Delete the subscription and the topic you created.

In [11]:
try:
    subscriber.delete_subscription(subscription=subscription_path)
except NotFound:
    pass
try:
    publisher.delete_topic(topic=topic_path)
except NotFound:
    pass

## What's next

- [Pub/Sub tools in ADK](https://adk.dev/integrations/pubsub/)
- [Agent Runtime overview](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) for running this agent as a managed service
- [Quickstart guide](README.md) for running this agent in the ADK developer UI
- Next quickstart: [Agent-to-agent delegation](../12-a2a-agent/walkthrough.ipynb)